In [1]:
import os
import numpy as np
import pandas as pd

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")



'1'

In [2]:
BASE_PATH_CANDIDATES = [
    "/kaggle/input/plant-pathology-2020-fgvc7",
    "/kaggle/data/plant-pathology-2020-fgvc7",
    "/kaggle/input",
    "/kaggle/data",
]
BASE_PATH = None
for p in BASE_PATH_CANDIDATES:
    if os.path.exists(os.path.join(p, "sample_submission.csv")) and os.path.exists(
        os.path.join(p, "test.csv")
    ):
        BASE_PATH = p
        break

if BASE_PATH is None:
    raise FileNotFoundError(
        "Could not locate sample_submission.csv and test.csv in expected Kaggle paths."
    )

SAMPLE_SUB_PATH = os.path.join(BASE_PATH, "sample_submission.csv")
TEST_CSV_PATH = os.path.join(BASE_PATH, "test.csv")
TRAIN_CSV_PATH = os.path.join(BASE_PATH, "train.csv")

IMAGES_DIR_CANDIDATES = [
    os.path.join(BASE_PATH, "images"),
    os.path.join(BASE_PATH, "plant-pathology-2020-fgvc7", "images"),
]
IMAGES_DIR = None
for d in IMAGES_DIR_CANDIDATES:
    if os.path.isdir(d):
        IMAGES_DIR = d
        break
if IMAGES_DIR is None:
    raise FileNotFoundError(
        "Could not locate images directory under expected Kaggle paths."
    )

print("Using BASE_PATH:", BASE_PATH)
print("SAMPLE_SUB_PATH:", SAMPLE_SUB_PATH)
print("TEST_CSV_PATH:", TEST_CSV_PATH)
print("TRAIN_CSV_PATH:", TRAIN_CSV_PATH)
print("IMAGES_DIR:", IMAGES_DIR)



Using BASE_PATH: /kaggle/input/plant-pathology-2020-fgvc7
SAMPLE_SUB_PATH: /kaggle/input/plant-pathology-2020-fgvc7/sample_submission.csv
TEST_CSV_PATH: /kaggle/input/plant-pathology-2020-fgvc7/test.csv
TRAIN_CSV_PATH: /kaggle/input/plant-pathology-2020-fgvc7/train.csv
IMAGES_DIR: /kaggle/input/plant-pathology-2020-fgvc7/images


In [3]:
SUBMISSIONS_PATH = "/kaggle/input/submissions/submissions/"

submissions_all = []
if os.path.exists(SUBMISSIONS_PATH):
    for dirname, _, filenames in os.walk(SUBMISSIONS_PATH):
        for filename in filenames:
            if filename.lower().endswith(".csv"):
                submissions_all.append(os.path.join(dirname, filename))
submissions_all.sort()
print("Found submission files:", submissions_all)




Found submission files: []


In [4]:
def _make_fallback_submissions(
    sample_sub_path: str, test_csv_path: str, train_csv_path: str, images_dir: str
) -> list:
    """
    Score improvement (toward target) with minimal semantic change:
    - Keep the same high-level approach: create 2 prediction CSVs -> ensemble.
    - Keep the same feature extraction core: 48x48 center-crop RGB + 6 simple color stats.
    - Keep the same model family and calibration method: LinearSVC + sigmoid calibration with cv=3.
    - Metric-aligned fix: since evaluation is mean column-wise ROC AUC (multi-label), train one calibrated
      binary classifier per target instead of forcing a noisy single multiclass label via argmax.
      This preserves the same training approach (SVM+calibration), but matches the metric and should
      raise AUC substantially toward the target.
    """
    from PIL import Image
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import LinearSVC
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.model_selection import StratifiedKFold
    import hashlib

    sample = pd.read_csv(sample_sub_path)
    test = pd.read_csv(test_csv_path)
    train = pd.read_csv(train_csv_path)

    required_cols = ["image_id", "healthy", "multiple_diseases", "rust", "scab"]
    missing = [c for c in required_cols if c not in sample.columns]
    if missing:
        raise ValueError(f"sample_submission.csv missing columns: {missing}")

    target_cols = ["healthy", "multiple_diseases", "rust", "scab"]
    for c in target_cols:
        if c not in train.columns:
            raise ValueError(f"train.csv missing required target column: {c}")

    test_ids = test["image_id"].astype(str).tolist()
    train_ids = train["image_id"].astype(str).tolist()

    sample = sample.set_index("image_id").reindex(test_ids).reset_index()
    if sample["image_id"].isna().any():
        raise ValueError(
            "sample_submission is missing some image_id entries present in test.csv."
        )

    Y = train[target_cols].to_numpy(dtype=np.int64, copy=True)

    _path_cache = {}

    def _resolve_path(img_id: str) -> str:
        p = _path_cache.get(img_id)
        if p is not None:
            return p

        direct = os.path.join(images_dir, img_id)
        if os.path.exists(direct):
            _path_cache[img_id] = direct
            return direct

        direct2 = os.path.join(images_dir, f"{img_id}.jpg")
        if os.path.exists(direct2):
            _path_cache[img_id] = direct2
            return direct2

        _path_cache[img_id] = ""
        return ""

    def _cache_paths(size: int, crop_frac: float, with_stats: bool):
        # NOTE: cache key version bumped because model training changed to 4 binary targets.
        tag_base = (
            f"pp2020_rgb{size}_crop{int(crop_frac*100)}_stats{int(with_stats)}_bin_v6"
        )
        h = hashlib.md5(os.path.abspath(images_dir).encode("utf-8")).hexdigest()[:10]
        xtr = os.path.abspath(f"{tag_base}_{h}_X_train.npy")
        xte = os.path.abspath(f"{tag_base}_{h}_X_test.npy")
        return xtr, xte

    def _center_crop(im: Image.Image, frac: float) -> Image.Image:
        if frac >= 0.999:
            return im
        w, h = im.size
        new_w = max(1, int(round(w * frac)))
        new_h = max(1, int(round(h * frac)))
        left = (w - new_w) // 2
        top = (h - new_h) // 2
        return im.crop((left, top, left + new_w, top + new_h))

    def featurize(image_ids, size=48, crop_frac=0.90, with_stats=True):
        base_dim = size * size * 3
        extra_dim = 6 if with_stats else 0
        X = np.zeros((len(image_ids), base_dim + extra_dim), dtype=np.float32)
        missing_count = 0

        img_open = Image.open
        bilinear = Image.BILINEAR
        asarray = np.asarray
        inv255 = np.float32(1.0 / 255.0)

        for i, img_id in enumerate(image_ids):
            img_path = _resolve_path(img_id)
            if not img_path:
                missing_count += 1
                continue
            with img_open(img_path) as im:
                im = im.convert("RGB")
                im = _center_crop(im, crop_frac).resize((size, size), resample=bilinear)
                arr = asarray(im, dtype=np.float32) * inv255  # (H,W,3) in [0,1]

            base_dim_local = base_dim
            X[i, :base_dim_local] = arr.reshape(-1)

            if with_stats:
                ch_mean = arr.mean(axis=(0, 1))  # (3,)
                ch_std = arr.std(axis=(0, 1))  # (3,)
                X[i, base_dim_local : base_dim_local + 3] = ch_mean
                X[i, base_dim_local + 3 : base_dim_local + 6] = ch_std

        if missing_count:
            print(
                f"Warning: {missing_count}/{len(image_ids)} images were missing under {images_dir}. They were featurized as zeros."
            )
        return X

    def _zero_row_fraction(X):
        Xv = np.asarray(X)
        return float(np.mean(np.all(Xv == 0.0, axis=1)))

    crop_frac = 0.90
    with_stats = True
    cache_train, cache_test = _cache_paths(
        size=48, crop_frac=crop_frac, with_stats=with_stats
    )

    if os.path.exists(cache_train) and os.path.exists(cache_test):
        X_train = np.load(cache_train, mmap_mode="r")
        X_test = np.load(cache_test, mmap_mode="r")

        ztr = _zero_row_fraction(X_train)
        zte = _zero_row_fraction(X_test)
        if (ztr > 0.05) or (zte > 0.05):
            print(
                f"Cached features look suspiciously sparse (all-zero rows: train={ztr:.3f}, test={zte:.3f}). Rebuilding features once."
            )
            X_train = featurize(
                train_ids, size=48, crop_frac=crop_frac, with_stats=with_stats
            )
            X_test = featurize(
                test_ids, size=48, crop_frac=crop_frac, with_stats=with_stats
            )
            np.save(cache_train, X_train)
            np.save(cache_test, X_test)
        else:
            print(
                f"Loaded cached features (all-zero rows: train={ztr:.3f}, test={zte:.3f})."
            )
    else:
        X_train = featurize(
            train_ids, size=48, crop_frac=crop_frac, with_stats=with_stats
        )
        X_test = featurize(
            test_ids, size=48, crop_frac=crop_frac, with_stats=with_stats
        )
        np.save(cache_train, X_train)
        np.save(cache_test, X_test)

    # Two slightly different base SVM pipelines (kept, to produce two complementary CSVs for the ensemble).
    base_svm1 = Pipeline(
        steps=[
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("svm", LinearSVC(C=2.0, class_weight=None, random_state=0, max_iter=5000)),
        ]
    )
    base_svm2 = Pipeline(
        steps=[
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("svm", LinearSVC(C=0.7, class_weight=None, random_state=1, max_iter=5000)),
        ]
    )

    # Keep cv=3 and sigmoid calibration; use StratifiedKFold for stability on imbalanced labels.
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    # --- Change for score improvement toward target:
    # Train one calibrated binary model per target (matches mean column-wise ROC AUC) instead of noisy multiclass argmax.
    alpha = 0.35  # keep the same prior-mixing behavior for degenerate labels
    P1 = np.zeros((X_test.shape[0], 4), dtype=np.float64)
    P2 = np.zeros((X_test.shape[0], 4), dtype=np.float64)

    for j, col in enumerate(target_cols):
        yj = Y[:, j].astype(np.int64, copy=False)
        prior = float(yj.mean())

        # If a column is degenerate (shouldn't happen here, but keep guard), fall back to smoothed prior.
        if yj.min() == yj.max():
            P1[:, j] = prior
            P2[:, j] = prior
            continue

        clf1 = CalibratedClassifierCV(base_svm1, method="sigmoid", cv=cv, n_jobs=-1)
        clf2 = CalibratedClassifierCV(base_svm2, method="sigmoid", cv=cv, n_jobs=-1)
        clf1.fit(X_train, yj)
        clf2.fit(X_train, yj)

        p1 = clf1.predict_proba(X_test)[:, 1].astype(np.float64, copy=False)
        p2 = clf2.predict_proba(X_test)[:, 1].astype(np.float64, copy=False)

        # Keep the same style of light prior mixing as a conservative stabilizer.
        P1[:, j] = (1.0 - alpha) * prior + alpha * p1
        P2[:, j] = (1.0 - alpha) * prior + alpha * p2

    np.clip(P1, 1e-6, 1 - 1e-6, out=P1)
    np.clip(P2, 1e-6, 1 - 1e-6, out=P2)

    sub1 = sample.copy()
    sub2 = sample.copy()
    sub1.loc[:, target_cols] = P1
    sub2.loc[:, target_cols] = P2

    sub1 = sub1.set_index("image_id").reindex(test_ids).reset_index()
    sub2 = sub2.set_index("image_id").reindex(test_ids).reset_index()

    path1 = "fallback_submission_1.csv"
    path2 = "fallback_submission_2.csv"
    sub1.to_csv(path1, index=False)
    sub2.to_csv(path2, index=False)
    return [os.path.abspath(path1), os.path.abspath(path2)]


if len(submissions_all) == 0:
    submissions_all = _make_fallback_submissions(
        SAMPLE_SUB_PATH, TEST_CSV_PATH, TRAIN_CSV_PATH, IMAGES_DIR
    )
    print("Created fallback submissions:", submissions_all)




/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

Created fallback submissions: ['/kaggle/working/fallback_submission_1.csv', '/kaggle/working/fallback_submission_2.csv']


/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [5]:
def ensemble(submissions_all, sub_idx, weights=[]):
    if len(sub_idx) == 0:
        raise ValueError("sub_idx must contain at least one index.")
    if len(submissions_all) == 0:
        raise ValueError(
            "submissions_all is empty; no submissions to ensemble. Fallback generation should have populated it."
        )
    if len(weights) == 0:
        weights = [1.0 / len(sub_idx)] * len(sub_idx)
    if len(weights) != len(sub_idx):
        raise ValueError(
            f"weights length ({len(weights)}) must match sub_idx length ({len(sub_idx)})."
        )

    cols = ["healthy", "multiple_diseases", "rust", "scab"]
    usecols = ["image_id"] + cols
    submission_sum = None
    base_len = None

    test_ids = (
        pd.read_csv(TEST_CSV_PATH, usecols=["image_id"])["image_id"]
        .astype(str)
        .tolist()
    )

    for i in range(len(sub_idx)):
        idx = sub_idx[i]
        if idx < 0 or idx >= len(submissions_all):
            raise IndexError(
                f"sub_idx[{i}]={idx} out of range for submissions_all (len={len(submissions_all)})."
            )

        w = float(weights[i])
        print(f"I'm taking submission {submissions_all[idx]} with weight {w}")
        submission = pd.read_csv(submissions_all[idx], usecols=usecols)

        missing_cols = [c for c in cols if c not in submission.columns]
        if missing_cols:
            raise ValueError(
                f"Submission {submissions_all[idx]} missing columns: {missing_cols}"
            )

        submission["image_id"] = submission["image_id"].astype(str)
        submission = submission.set_index("image_id").reindex(test_ids).reset_index()
        if submission["image_id"].isna().any():
            raise ValueError(
                f"Submission {submissions_all[idx]} is missing some image_ids from test.csv"
            )

        arr = submission.loc[:, cols].to_numpy(dtype=np.float64, copy=False)

        if base_len is None:
            base_len = arr.shape[0]
            submission_sum = np.zeros_like(arr, dtype=np.float64)
        elif arr.shape[0] != base_len:
            raise ValueError(
                f"Row count mismatch across submissions: expected {base_len}, got {arr.shape[0]}"
            )

        submission_sum += arr * w

    return submission_sum




In [6]:
def make_submission_file(submission_avg, submissions_all):
    submission_df = pd.read_csv(
        SAMPLE_SUB_PATH,
        usecols=["image_id", "healthy", "multiple_diseases", "rust", "scab"],
    )
    test_df = pd.read_csv(TEST_CSV_PATH, usecols=["image_id"])

    test_ids = test_df["image_id"].astype(str).tolist()
    submission_df["image_id"] = submission_df["image_id"].astype(str)
    submission_df = submission_df.set_index("image_id").reindex(test_ids).reset_index()
    if submission_df["image_id"].isna().any():
        raise ValueError(
            "sample_submission is missing some image_id entries present in test.csv."
        )

    cols = ["healthy", "multiple_diseases", "rust", "scab"]
    if submission_avg.shape != (len(submission_df), len(cols)):
        raise ValueError(
            f"submission_avg has shape {submission_avg.shape}, expected {(len(submission_df), len(cols))}"
        )

    submission_df.loc[:, cols] = submission_avg
    submission_df[cols] = submission_df[cols].clip(0.0, 1.0)

    submission_df.to_csv("submission.csv", index=False)
    print("Wrote submission.csv with shape:", submission_df.shape)
    print("Columns:", list(submission_df.columns))
    print(submission_df.head())


submission_avg = ensemble(submissions_all, [0, 1], [0.24, 0.76])
make_submission_file(submission_avg, submissions_all)

I'm taking submission /kaggle/working/fallback_submission_1.csv with weight 0.24
I'm taking submission /kaggle/working/fallback_submission_2.csv with weight 0.76
Wrote submission.csv with shape: (183, 5)
Columns: ['image_id', 'healthy', 'multiple_diseases', 'rust', 'scab']
  image_id   healthy  multiple_diseases      rust      scab
0   Test_0  0.263089           0.053179  0.343853  0.312901
1   Test_1  0.278471           0.053843  0.338816  0.356340
2   Test_2  0.262073           0.053780  0.319761  0.354277
3   Test_3  0.270316           0.051639  0.340549  0.330235
4   Test_4  0.252096           0.053421  0.320501  0.368790
